# ARTERY Feedback Tutorial

This notebook walks through the software analysis path used by the ARTERY-style feedback demo: loading readout data, demodulating IQ traces, classifying |0>/|1> states, building early trajectory predictors, and generating feedback waveforms.

## 1. Data Loading and Dataset Structure
Load the readout dataset, inspect its metadata, and confirm the tensor layout used by the remaining analysis cells.

In [ ]:
from pathlib import Path
import gzip
import shutil

# The Docker tutorial image stores the dataset as tutorial/readout_data.mat.gz.
# The original analysis cells expect ./readout_data.mat in the current workdir.
data_path = Path('./readout_data.mat')
compressed_candidates = [
    Path('./tutorial/readout_data.mat.gz'),
    Path('./readout_data.mat.gz'),
    Path('../tutorial/readout_data.mat.gz'),
]
if not data_path.exists():
    for compressed in compressed_candidates:
        if compressed.exists():
            with gzip.open(compressed, 'rb') as fin, data_path.open('wb') as fout:
                shutil.copyfileobj(fin, fout)
            break
if not data_path.exists():
    raise FileNotFoundError('readout_data.mat was not found, and no readout_data.mat.gz candidate could be extracted.')
print(f'Using readout data: {data_path.resolve()}')


In [ ]:
import scipy.io as sio
loadmat = sio.loadmat
read_data = loadmat('./readout_data.mat')
print(read_data.keys())
print(read_data['data'].shape)
print(read_data['data_info_name'])
print(read_data['qubits'])
print(read_data['state'])
print(read_data['stats'])
print(read_data['delay'])
print(read_data['measure_fids'])
print(read_data['data'][0][0][0])

**Shape: (2, 2000, 4096, 2)<br>Two prepared states:|0> and |1>; are each repeated 2,000 times; Each waveform contains 4,096 sampling points across both the I and Q channels.**<br>
**Qubits: [q1.q2,q3]**


## 2. IQ Readout Visualization and 0/1 State Separation
Plot representative I/Q traces and compare the average readout response of prepared |0> and |1> states.

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
read_zero = read_data['data'][0][:][:][:]
read_zero_i = read_zero[:, :, 0]
print(read_zero_i.shape)
print(read_zero[0][1])
read_zero_q = read_zero[:, :, 1]
read_one = read_data['data'][1][:][:][:]
read_one_i = read_one[:, :, 0]
read_one_q = read_one[:, :, 1]
plt.figure(figsize=(200,40))
plt.plot(np.mean(read_zero_q,axis=0), color='black', linewidth=30.0)
plt.plot(np.mean(read_one_q,axis=0))

new = np.mean(read_zero_q, axis=0)
plt.figure(figsize=(200,40))
plt.plot(new[500:1750], color='black', linewidth=40.0)


The first figure is the averaged Q-channel waveform for state 0 over all 4096 points, and the second figure is the averaged Q-channel waveform for state 0 over a 2‑microsecond duration.<br>The most distinctive waveforms are only a fraction of all waveforms.

## 3. Why Demodulation Is Needed
Show that raw energy alone cannot reliably separate |0> and |1>, while the time-domain waveform carries state-dependent phase information.

In [ ]:
energy_zero = np.array([np.sum(read_zero_i[_] ** 2) for _ in range(read_zero_i.shape[0])])
energy_one = np.array([np.sum(read_one_i[_] ** 2) for _ in range(read_one_i.shape[0])])
print(energy_one)
print(f"energy of wave 0 is {np.mean(energy_zero)} and top 10 is {np.sort(energy_zero)[::-1][:10]} and energy of wave 1 is {np.mean(energy_one)} and top 10 is {np.sort(energy_one)[::-1][:10]}")
plt.plot(np.mean(read_zero_q, axis=0))
plt.plot(np.mean(read_one_q, axis=0))

Both the average energy and the top ten largest individual energies fail to distinguish between state 0 and state 1.<br>The 0 and 1 states are distinguishable in the waveforms, and the distinguishability is concentrated in a certain time window


## 4. IQ Demodulation / Digital Down-Conversion
Convert each raw readout trace into an integrated IQ point by mixing with the readout frequency and summing over a selected time window.

**Demodulation formula (digital down‑conversion) **

$$
\begin{aligned}
I &= \sum_{t}\big[\,I_{\text{raw}}(t)\cos(\omega t+\varphi) + Q_{\text{raw}}(t)\sin(\omega t+\varphi)\,\big]\\
Q &= \sum_{t}\big[\,Q_{\text{raw}}(t)\cos(\omega t+\varphi) - I_{\text{raw}}(t)\sin(\omega t+\varphi)\,\big]
\end{aligned}
$$

Complex‑form：

$$
S=\sum_{t}\big(I_{\text{raw}}(t)+j\,Q_{\text{raw}}(t)\big)\,e^{-j\omega t},\qquad I=\operatorname{Re}(S),\quad Q=\operatorname{Im}(S)
$$



In [ ]:
def func_0(omega, read_i, read_q, phase=0):
    assert read_i.shape == read_q.shape
    ts = np.arange(0,read_i.shape[1])
    cos_ = np.array([np.cos(omega * ts + phase)]*read_i.shape[0])
    sin_ = np.array([np.sin(omega * ts + phase)]*read_i.shape[0])
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return sum_i, sum_q

def func_1(omega, read_i, read_q, phase=0):
    assert read_i.shape == read_q.shape
    read_data = read_i + 1j*read_q
    ts = np.arange(0,read_data.shape[1])
    mix_data = np.array([np.exp(-1j*omega*ts+phase)]*read_data.shape[0])
    sum_data = np.sum(read_data*mix_data, axis=1)
    return sum_data.real, sum_data.imag


omega = 2*np.pi*(np.array([6.881, 6.79525, 6.97284])-7)
idx1 = 1
idx2 = 2000
result_zero = func_0(omega[2], read_zero_i[idx1:idx2], read_zero_q[idx1:idx2])
result_one = func_0(omega[2], read_one_i[idx1:idx2], read_one_q[idx1:idx2])
# plt.figure()
plt.scatter(result_zero[0], result_zero[1])
plt.scatter(result_one[0], result_one[1])
plt.show()

In demodulation, the entire time‑domain waveform of each shot is compressed into a single point.<br>The 0 and 1 states fall at different positions/angles on the IQ plane due to the dispersion‑induced phase shift — and this is precisely the discriminative information that allows the two states to be distinguished, while energy cannot

## 5. K-Means Clustering for State Classification
Use the demodulated IQ points to form two clusters and map them to the |0> and |1> measurement outcomes.

In [ ]:
result = np.c_[(np.array(result_zero)), (np.array(result_one))].T
from sklearn.cluster import KMeans
from sklearn import metrics
kmeans = KMeans(n_clusters=2, random_state=0).fit(result)
print(metrics.calinski_harabasz_score(result, kmeans.labels_))
print(f"Center 0 = {kmeans.cluster_centers_[0]}; and Center 1 = {kmeans.cluster_centers_[1]}")
print(kmeans.score)
plt.scatter(result[:, 0], result[:, 1], c=kmeans.labels_, cmap='viridis')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=100, c='red')  # 绘制聚类中心
plt.title('K-means Clustering')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

Based on historical data, unsupervised K‑means clustering forms two clusters corresponding to the 0 and 1 states, with two centroids.<br>
According to the obvious dividing line in the middle, the points can be separated into states 0 and 1.

## 6. IQ Trajectory and Segmented Demodulation
Track how each shot moves on the IQ plane as the integration window grows, exposing when the state information becomes distinguishable.

The figure below illustrates the core segmented-readout idea used by ARTERY. A readout pulse is not treated as one indivisible block; instead, partial integration windows are evaluated as the pulse arrives. Each longer window gives a new IQ point, so the hardware can observe a trajectory instead of waiting for the full readout to finish.

![Segmented readout pulse and IQ trajectory](results/3_2_readout_segmented_trajectory.png)

In the implementation, these partial IQ points become trajectory features. Early windows are noisy, but later windows move toward the state-dependent IQ cluster, which makes confidence-based early feedback possible.


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib as mpl


def demod(omega, read_i, read_q, tstep=0, phase=0):
    assert read_i.shape == read_q.shape
    ts = np.arange(0, (tstep+1)* window_len)
    cos_ = np.array([np.cos(omega * ts + phase)]*read_i.shape[0])
    sin_ = np.array([np.sin(omega * ts + phase)]*read_i.shape[0])
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return sum_i[0], sum_q[0]

window_base, window_cnt, window_len, shots, base_shot = 850, 6, 300, 2, 122
omega = 2*np.pi*(np.array([6.881, 6.79525, 6.97284])-7)
result_zero, result_one = [[0 for _ in range(window_cnt)] for _ in range(shots)], [[0 for _ in range(window_cnt)] for _ in range(shots)]

for shot in range(shots):
    for tstep in range(0, window_cnt):
        result_zero[shot][tstep] = demod(omega[0], read_zero_i[base_shot + shot - 1][window_base : window_base + (tstep+1) * window_len], read_zero_q[base_shot + shot - 1][window_base : window_base + (tstep+1) * window_len], tstep)
        result_one[shot][tstep] = demod(omega[0], read_one_i[base_shot + shot][window_base : window_base + (tstep+1) * window_len], read_one_q[base_shot + shot][window_base : window_base + (tstep+1) * window_len], tstep)
size = np.arange(window_cnt) * 50
size = np.clip(size, 10, 500)

for shot in range(0, shots):
    result_zero_arr, result_one_arr = np.array(result_zero[shot]), np.array(result_one[shot])
    if shot == 0:
        plt.scatter(result_zero_arr[:, 0], result_zero_arr[:, 1], s=size, color='skyblue', label="trajectory of |1>")
        plt.plot(result_zero_arr[:, 0], result_zero_arr[:, 1], color='skyblue', linestyle='-.', linewidth=2.5)
        plt.scatter(result_one_arr[:, 0], result_one_arr[:, 1], s=size, color='orange', label="trajectory of |0>")
        plt.plot(result_one_arr[:, 0], result_one_arr[:, 1], color='orange', linestyle='-.', linewidth=2.5)
    else:
        plt.scatter(result_zero_arr[:, 0], result_zero_arr[:, 1], s=size, color='skyblue')
        plt.plot(result_zero_arr[:, 0], result_zero_arr[:, 1], color='skyblue', linestyle='-.', linewidth=2.5)
        plt.scatter(result_one_arr[:, 0], result_one_arr[:, 1], s=size, color='orange')
        plt.plot(result_one_arr[:, 0], result_one_arr[:, 1], color='orange', linestyle='-.', linewidth=2.5)

    # plt.plot(result_zero[:, 0], result_zero[:, 1], s=size, marker='o', linestyle='-.', color='blue', label='state_0')
    # plt.plot(result_one[:, 0], result_one[:, 1], s=size, marker='o', linestyle='-.', color='red', label='state_1')
plt.legend(loc='upper left', handlelength=0.6, borderaxespad=0.2, borderpad=0.2, labelspacing=0.5, handletextpad=0.2, prop={'family': 'DejaVu Serif', 'size': 30})
my_x_ticks = np.arange(-800, 401, 400)
my_y_ticks = np.arange(-500, 1501, 500)
plt.xticks(my_x_ticks, fontproperties = 'DejaVu Serif', fontsize=26)
plt.yticks(my_y_ticks, fontproperties = 'DejaVu Serif', fontsize=26)
# pdf = PdfPages('trace.pdf')
# plt.savefig('trace.pdf', format='pdf')
plt.show()


$(I+jQ)e^{-j\omega t}$Fixed window start (window_base = 850), with the integration window length gradually increased (point size ∝ window length). The IQ point of each shot moves along a trajectory, and the centroids of the |0⟩ and |1⟩ states become increasingly separated on the IQ plane.

Physical reason: Demodulation is essentially a time integration of
$(I+jQ)e^{-j\omega t}$ over the window. A longer window means: ① more accumulated dispersive phase difference between the two states; ② more coherent summation of the useful signal and better averaging of random noise, leading to an improved SNR. Hence the separation increases.

## 7. First Segmented Demodulation Baseline
Evaluate an early, short readout window to establish the low-latency but low-accuracy baseline.

In [ ]:
def demod_part(omega, read_i, read_q, phase=0):
    assert read_i.shape == read_q.shape
    print(f"read_i shape is {read_i.shape}")
    ts = np.arange(0,read_i.shape[1])
    cos_ = np.array([np.cos(omega * ts + phase)]*read_i.shape[0])
    sin_ = np.array([np.sin(omega * ts + phase)]*read_i.shape[0])
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return sum_i, sum_q


omega = 2*np.pi*(np.array([6.881, 6.79525, 6.97284])-7)
idx1, idx2 = 0, 2000
window_start, window_len = 850, 200

result_zero = demod_part(omega[0], read_zero_i[idx1:idx2, window_start:window_start+window_len], read_zero_q[idx1:idx2, window_start:window_start+window_len])
result_one = demod_part(omega[0], read_one_i[idx1:idx2, window_start:window_start+window_len], read_one_q[idx1:idx2, window_start:window_start+window_len])
# plt.figure()
plt.scatter(result_zero[0], result_zero[1])
plt.scatter(result_one[0], result_one[1])

result = np.c_[(np.array(result_zero)), (np.array(result_one))].T
true_labels = np.array([0]*result_zero[0].shape[0] + [1]*result_one[0].shape[0])
print(f"shape of true_label: {true_labels.shape} and 20 items of true_label: {true_labels[:20]}")
from sklearn.cluster import KMeans
from sklearn import metrics
pred = KMeans(n_clusters=2, random_state=0).fit(result)
print(metrics.calinski_harabasz_score(result, pred.labels_))
print(f"Center 0 = {kmeans.cluster_centers_[0]}; and Center 1 = {kmeans.cluster_centers_[1]}")
# print(pred.score(result))

pred_labels = pred.labels_
accuracy = metrics.accuracy_score(true_labels, pred_labels)
print(f"Accuracy: {accuracy}")


plt.scatter(result[true_labels == 0][pred_labels[true_labels == 0] == 0, 0], result[true_labels == 0][pred_labels[true_labels == 0] == 0, 1], c='lightskyblue', label='0 to 0') # 正确分类的0
plt.scatter(result[true_labels == 0][pred_labels[true_labels == 0] == 1, 0], result[true_labels == 0][pred_labels[true_labels == 0] == 1, 1], c='navajowhite', label='0 to 1') # 0错分成1
plt.scatter(result[true_labels == 1][pred_labels[true_labels == 1] == 1, 0], result[true_labels == 1][pred_labels[true_labels == 1] == 1, 1], c='orange', label='1 to 1') # 正确分类的1
plt.scatter(result[true_labels == 1][pred_labels[true_labels == 1] == 0, 0], result[true_labels == 1][pred_labels[true_labels == 1] == 0, 1], c='dodgerblue', label='1 to 0') # 1错分成0

# 绘制聚类中心
plt.scatter(pred.cluster_centers_[0, 0], pred.cluster_centers_[0, 1], s=100, c='red', label='Center 0')
plt.scatter(pred.cluster_centers_[1, 0], pred.cluster_centers_[1, 1], s=100, c='indigo', label='Center 1')
plt.title('K-means Clustering for Measurements')
# plt.xlabel('X-axis')
# plt.ylabel('Y-axis')
plt.legend()
plt.savefig('cluster.pdf', format='pdf')
plt.show()

# plt.scatter(result[true_labels == 0][pred_labels == 0, 0], result[true_labels == 0][pred_labels == 0, 1], c='blue', label='0 (Correctly Classified)') # 正确分类的0
# plt.scatter(result[true_labels == 0][pred_labels == 1, 0], result[true_labels == 0][pred_labels == 1, 1], c='orange', label='0 (Misclassified as 1)') # 0错分成1
# plt.scatter(result[true_labels == 1][pred_labels == 1, 0], result[true_labels == 1][pred_labels == 1, 1], c='green', label='1 (Correctly Classified)') # 正确分类的1
# plt.scatter(result[true_labels == 1][pred_labels == 0, 0], result[true_labels == 1][pred_labels == 0, 1], c='purple', label='1 (Misclassified as 0)') # 1错分成0
# plt.scatter(pred.cluster_centers_[:, 0], pred.cluster_centers_[:, 1], s=100, c='red', label='Centroids') # 绘制聚类中心

# plt.title('K-means Clustering with True Labels')
# plt.xlabel('X-axis')
# plt.ylabel('Y-axis')
# plt.legend()
# plt.show()

With window_start = 850 and window_len = 200, the supervised K‑means clustering yields a classification accuracy of only 57.5%. We will subsequently search for a better window.

## 8. Accuracy vs. Readout Window Start
Sweep the start point of the demodulation window to reveal the accuracy-latency trade-off in the readout trajectory.

In [ ]:
import matplotlib as mpl
mpl.rcParams['text.usetex'] = False
pred = np.array([0.605, 0.6895, 0.773, 0.822, 0.823, 0.857, 0.8915, 0.906, 0.9205, 0.933])
plt.scatter(np.arange(850, 2700, 200), pred, s=np.arange(1, 11) * 10, color='blue', label='demod acc')
plt.plot(np.arange(850, 2700, 200),pred, color='black', linestyle='-.')
plt.show()

Fixed window length, with the window start point gradually shifted backward. The K‑means classification accuracy increases monotonically from 0.605 to 0.933.<br>
Physical reason: The phase difference in the cavity response induced by the |0⟩ and |1⟩ states accumulates over time. In later time segments, the two states become increasingly separated on the IQ plane, yielding stronger distinguishable information — consistent with the earlier observation that "distinguishability is concentrated in a certain time window."<br>
Cost: Shifting the start point further backward means a longer waiting time, implying a trade‑off between classification accuracy and feedback latency. This is precisely the motivation for the subsequent search for the "optimal window / shortest latency that achieves the target accuracy."

In [ ]:
def demod_part(omega, read_i, read_q, phase=0):
    assert read_i.shape == read_q.shape
    print(f"read_i shape is {read_i.shape}")
    ts = np.arange(0,read_i.shape[1])
    cos_ = np.array([np.cos(omega * ts + phase)]*read_i.shape[0])
    sin_ = np.array([np.sin(omega * ts + phase)]*read_i.shape[0])
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return sum_i, sum_q


omega = 2*np.pi*(np.array([6.881, 6.79525, 6.97284])-7)
idx1, idx2 = 0, 1000
window_start, window_len = 850, 1800

result_zero = demod_part(omega[2], read_zero_i[idx1:idx2, window_start:window_start+window_len], read_zero_q[idx1:idx2, window_start:window_start+window_len])
result_one = demod_part(omega[2], read_one_i[idx1:idx2, window_start:window_start+window_len], read_one_q[idx1:idx2, window_start:window_start+window_len])

result_zero, result_one = np.array(result_zero).T, np.array(result_one).T
print(f"zero shape is {result_zero.shape}, one shape is {result_one.shape}")

center_zero, center_one = result_zero.mean(axis=0), result_one.mean(axis=0)
print(f"pred_zero is {center_zero}, pred_one is {center_one}")

idx1, idx2 = 1000, 2000
test_zero = demod_part(omega[2], read_zero_i[idx1:idx2, window_start:window_start+window_len], read_zero_q[idx1:idx2, window_start:window_start+window_len])
test_one = demod_part(omega[2], read_one_i[idx1:idx2, window_start:window_start+window_len], read_one_q[idx1:idx2, window_start:window_start+window_len])
test_data = np.c_[(np.array(test_zero)), (np.array(test_one))].T
test_label = np.array([0]*(np.array(test_zero).shape[1]) + [1]*(np.array(test_one).shape[1]))
print(test_label.shape)
test_result = np.zeros(test_label.shape[0])
for i in range(test_label.shape[0]):
    test_result[i] = 0 if np.linalg.norm(test_data[i] - center_zero) < np.linalg.norm(test_data[i] - center_one) else 1
print(f"Prediction Accuracy = {np.sum(test_result == test_label)/len(test_label)}")


For window_start = 850 and window_len = 1800, the accuracy ceiling is 0.933.

## 9. Trajectory Feature Construction
Convert growing-window IQ trajectories into compact binary location and heading features for the hardware-friendly predictor.

In [ ]:
from scipy.io import loadmat
import numpy as np
from sklearn.cluster import KMeans
from sklearn import metrics
import json

omegas = 2*np.pi*(np.array([6.881, 6.79525, 6.97284])-7)
read_data = loadmat('./readout_data.mat')
read_zero, read_one = read_data['data'][0][:][:][:], read_data['data'][1][:][:][:]
read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]

data_len, partition_zero, partition_one = read_zero_i.shape[0], 1, 0.25
read_zero_i_part, read_zero_q_part = read_zero_i[:int(data_len*partition_zero), :], read_zero_q[:int(data_len*partition_zero), :]
read_one_i_part, read_one_q_part = read_one_i[:int(data_len*partition_one), :], read_one_q[:int(data_len*partition_one), :]
train_div, records = 0.7, []
window_start, pred_step = 850, 25
pred_prob = partition_one / (partition_zero + partition_one)  # probability of measured as '1'
train_zero_idx1, test_zero_idx1 = 0, int(data_len * partition_zero * train_div)
train_zero_idx2, test_zero_idx2 = train_zero_idx1 + test_zero_idx1, int(data_len * partition_zero)
train_one_idx1, test_one_idx1 = 0, int(data_len * partition_one * train_div)
train_one_idx2, test_one_idx2 = train_one_idx1 + test_one_idx1, int(data_len * partition_one)

# To construct train-pattern table
train_pattern_table = []

# def write_list_to_json(list, json_file_name):
#     with open(json_file_name, 'w') as  f:
#         json.dump(list, f)

def demod_part(omega, read_i, read_q, phase=0):
    assert read_i.shape == read_q.shape
    ts = np.arange(0, read_i.shape[1])
    cos_ = np.array([np.cos(omega * ts + phase)]*read_i.shape[0])
    sin_ = np.array([np.sin(omega * ts + phase)]*read_i.shape[0])
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return sum_i, sum_q 

for omega in omegas:
    record = []
    for window_len in range(pred_step, 2001, pred_step):
        if window_start + window_len > 4096: break
        train_zero = demod_part(omega, read_zero_i_part[train_zero_idx1:train_zero_idx2, window_start:window_start+window_len], \
                                 read_zero_q_part[train_zero_idx1:train_zero_idx2, window_start:window_start+window_len])
        train_one = demod_part(omega, read_one_i_part[train_one_idx1:train_one_idx2, window_start:window_start+window_len], \
                                 read_one_q_part[train_one_idx1:train_one_idx2, window_start:window_start+window_len])
        result_zero, result_one = np.array(train_zero).T, np.array(train_one).T
        center_zero, center_one = result_zero.mean(axis=0), result_one.mean(axis=0)
        train_data = np.c_[(np.array(train_zero)), (np.array(train_one))].T
        train_label = np.array([0]*(np.array(train_zero).shape[1]) + [1]*(np.array(train_one).shape[1]))
        train_diff, train_location, train_heading = np.zeros(train_label.shape[0]), np.zeros(train_label.shape[0]), np.zeros(train_label.shape[0])
        for i in range(train_label.shape[0]):
            diff_new = np.linalg.norm(train_data[i] - center_zero) - np.linalg.norm(train_data[i] - center_one)
            train_location[i] = 0 if diff_new < 0 else 1
            if window_len > pred_step:
                train_heading[i] = 0 if diff_new < train_diff[i] else 1
            train_diff[i] = diff_new
        train_location_dots = train_location if window_len == pred_step else np.vstack((train_location_dots, train_location))
        if window_len > pred_step:
            train_heading_dots = train_heading if window_len == pred_step * 2 else np.vstack((train_heading_dots, train_heading))

        test_zero = demod_part(omega, read_zero_i_part[test_zero_idx1:test_zero_idx2, window_start:window_start+window_len], \
                               read_zero_q_part[test_zero_idx1:test_zero_idx2, window_start:window_start+window_len])
        test_one = demod_part(omega, read_one_i_part[test_one_idx1:test_one_idx2, window_start:window_start+window_len], \
                               read_one_q_part[test_one_idx1:test_one_idx2, window_start:window_start+window_len])
        test_data = np.c_[(np.array(test_zero)), (np.array(test_one))].T
        test_label = np.array([0]*(np.array(test_zero).shape[1]) + [1]*(np.array(test_one).shape[1]))
        test_diff, test_location, test_heading = np.zeros(test_label.shape[0]), np.zeros(test_label.shape[0]), np.zeros(test_label.shape[0])
        for i in range(test_label.shape[0]):
            diff_new = np.linalg.norm(test_data[i] - center_zero) - np.linalg.norm(test_data[i] - center_one)
            test_location[i] = 0 if diff_new < 0 else 1
            if window_len > pred_step:
                test_heading[i] = 0 if diff_new < test_diff[i] else 1
            test_diff[i] = diff_new
        test_location_dots = test_location if window_len == pred_step else np.vstack((test_location_dots, test_location))
        if window_len > pred_step:
            test_heading_dots = test_heading if window_len == pred_step * 2 else np.vstack((test_heading_dots, test_heading))
        
        # no more acc test in this block, leave it to the next block
        # accuracy = np.sum(test_result == test_label)/len(test_label)
        # record.append([window_len, accuracy])
    mulfreq_train_location_dots = train_location_dots if omega == omegas[0] else np.dstack((mulfreq_train_location_dots, train_location_dots))
    mulfreq_train_heading_dots = train_heading_dots if omega == omegas[0] else np.dstack((mulfreq_train_heading_dots, train_heading_dots))
    mulfreq_test_location_dots = test_location_dots if omega == omegas[0] else np.dstack((mulfreq_test_location_dots, test_location_dots))
    mulfreq_test_heading_dots = test_heading_dots if omega == omegas[0] else np.dstack((mulfreq_test_heading_dots, test_heading_dots))
    # record_sorted = sorted(record, key=lambda x:x[1], reverse=True)
    # records.append((omega, record_sorted))

# omegas_part, records_sorted_part = [item[0] for item in records], [item[1] for item in records]
# print(records_sorted_part)
print(mulfreq_train_location_dots.shape)
print(mulfreq_test_heading_dots.shape)
# write_list_to_json(best_selection, "part.json")

Each readout pulse is demodulated over a sequence of **growing windows** (step = `pred_step`), turning it into a *trajectory* of two per-step binary features: **location** (which centroid, |0> or |1>, the demodulated IQ point is closest to) and **heading** (whether the |0>/|1> distance gap is moving toward |1> versus the previous, shorter window). This is repeated for the 3 readout frequencies and stacked into the multi-frequency tensors `mulfreq_*_dots`, which the predictor (Module 10) consumes.

## 10. Branch Predictor Design
Build BHT/HHT-style history tables and use Bayesian confidence updates to decide whether early feedback can be issued.

ARTERY combines two sources of information: historical branch behavior from previous shots and the current shot trajectory. The history table provides a prior branch probability, while the current IQ trajectory provides a readout-driven probability. Bayesian fusion then produces a confidence value used for early branch selection.

![Bayesian branch prediction with history and current-shot trajectory](results/4_1_bayesian_branch_prediction.png)

This is the reason the tutorial separates trajectory extraction from branch prediction. The trajectory analyzer estimates the current measurement outcome, the branch history table stores repeated program behavior, and the Bayesian predictor decides whether the confidence is high enough to issue feedback before the full readout completes.


In [ ]:
#TODO: to consturct the location table and the heading table, calculate pred acc

# shot_front, shot_end = 1000, 1005
# print(mulfreq_train_location_dots[:, shot_front:shot_end, 0])
# print(mulfreq_train_heading_dots[:, shot_front:shot_end, 0])
# print(train_label[shot_front:shot_end])
trace_train_location = mulfreq_train_location_dots.transpose(2, 1, 0)
trace_train_heading = mulfreq_train_heading_dots.transpose(2, 1, 0)
trace_test_location = mulfreq_test_location_dots.transpose(2, 1, 0)
trace_test_heading = mulfreq_test_heading_dots.transpose(2, 1, 0)
print(trace_train_location[0, 0, :])

# to set the n-bit saturating counter and branch history table of probability
BHT_count, HHT_count, st_counters = {}, {}, 8
register_prob = trace_train_location[0, 0, 0:st_counters]
register_key = ''.join('1' if x != 0.0 else '0' for x in register_prob)
print(register_key)

for i in range(trace_train_location.shape[1]):
    for j in range(trace_train_location.shape[2] - st_counters):
        # to set the location tables
        register_prob = trace_train_location[0, i, j:j+st_counters]
        register_key = ''.join('1' if x != 0.0 else '0' for x in register_prob)
        if register_key not in BHT_count:
            if int(train_label[i]) == 0:
                zeros, ones = 1, 0
            else:
                zeros, ones = 0, 1
            BHT_count[register_key] = [zeros, ones]
        else:
            if int(train_label[i]) == 0:
                BHT_count[register_key][0] += 1
            else:
                BHT_count[register_key][1] += 1

        # to set the heading tables
        if j > 0:
            register_prob = trace_train_heading[0, i, j-1:j+st_counters-1]
            register_key = ''.join('1' if x != 0.0 else '0' for x in register_prob)
            if register_key not in HHT_count:
                if int(train_label[i]) == 0:
                    zeros, ones = 1, 0
                else:
                    zeros, ones = 0, 1
                HHT_count[register_key] = [zeros, ones]
            else:
                if int(train_label[i]) == 0:
                    HHT_count[register_key][0] += 1
                else:
                    HHT_count[register_key][1] += 1
BHT_prob = {key: count[1] / (count[0] + count[1]) for key, count in BHT_count.items()}
HHT_prob = {key: count[1] / (count[0] + count[1]) for key, count in HHT_count.items()}

theta = 0.91
ns_per_sample = 2000 / 4096
set_start_points = 0
set_len = trace_test_location.shape[2]          # use all available steps
n_steps = set_start_points + (set_len - st_counters)

origin_prob = partition_one / (partition_zero + partition_one)
test_result = np.zeros(trace_test_location.shape[1])
latency_ns  = np.zeros(trace_test_location.shape[1])
pred_probs, pred_heading_probs = [], []

def bayes(prior, obs):
    if obs in (0.0, 1.0): return obs
    return (obs * prior) / (obs * prior + (1 - obs) * (1 - prior))

def fuse(p, h):
    return (p * h) / (p * h + (1 - p) * (1 - h)) if p > 0 and h > 0 else max(p, h)

for i in range(trace_test_location.shape[1]):
    prob_per_shot, heading_per_shot = np.zeros(n_steps), np.zeros(n_steps)
    pred_prob = pred_heading = origin_prob
    decided = False
    for j in range(n_steps):
        register_key = ''.join('1' if x != 0.0 else '0'
                               for x in trace_test_location[0, i, j:j+st_counters])
        if j > 0:
            register_heading_key = ''.join('1' if x != 0.0 else '0'
                                   for x in trace_test_heading[0, i, j-1:j+st_counters-1])
            if register_heading_key in HHT_prob:           # bug fix: use heading key
                pred_heading = bayes(pred_heading, HHT_prob[register_heading_key])
            heading_per_shot[j-1] = pred_heading
        if register_key in BHT_prob:
            pred_prob = bayes(pred_prob, BHT_prob[register_key])
        prob_per_shot[j] = pred_prob

        fused = fuse(pred_prob, pred_heading)
        if not decided and (fused >= theta or fused <= 1 - theta):
            test_result[i] = 0 if fused < 0.5 else 1
            latency_ns[i] = (window_start + (j + st_counters) * pred_step) * ns_per_sample
            decided = True
    if not decided:                                        # never crossed: final decision + full-window latency
        fused = fuse(pred_prob, pred_heading)
        test_result[i] = 0 if fused < 0.5 else 1
        latency_ns[i] = (window_start + (n_steps - 1 + st_counters) * pred_step) * ns_per_sample
    pred_probs.append(prob_per_shot)
    pred_heading_probs.append(heading_per_shot)

accuracy = np.sum(test_result == test_label) / len(test_label)
print(f"acc = {accuracy}")
print(f"average feedback latency = {latency_ns.mean():.1f} ns ({latency_ns.mean()/1000:.3f} us)")
print(f"latency min/max = {latency_ns.min():.1f}/{latency_ns.max():.1f} ns")


A **branch predictor** for the readout, inspired by CPU branch prediction. From the training trajectories it builds two history tables keyed by the last `st_counters` steps: **BHT** for location patterns and **HHT** for heading patterns, each storing `P(measured = 1 | pattern)`. At inference, every test shot fuses the location and heading evidence with a step-by-step **Bayesian update**; as soon as the fused probability crosses the threshold `theta` (= 0.91), the branch is decided (pre-execution) and that shot’s **feedback latency** is recorded. It outputs the accuracy and the average feedback latency.

## 11. Best Demodulation Window Search
Sweep window lengths and select the lowest-latency configuration that still reaches the target prediction accuracy.

The search step exposes the hardware tradeoff: shorter windows reduce feedback latency, but can lose classification accuracy; longer windows improve confidence, but delay the feedback operation. The plot below shows this accuracy/latency balance across representative circuit workloads and window lengths.

![Accuracy and latency versus readout window length](results/4_2_accuracy_latency_comparison.png)

For a tutorial run, this section therefore chooses a practical window configuration instead of blindly using the shortest possible window. The selected point should preserve high prediction accuracy while keeping the feedback path short enough for low-latency control.


In [ ]:
import numpy as np
from scipy.io import loadmat

omegas = 2*np.pi*(np.array([6.881, 6.79525, 6.97284]) - 7)
_rd = loadmat('./readout_data.mat')
_z, _o = _rd['data'][0][:][:][:], _rd['data'][1][:][:][:]
read_zero_i, read_zero_q = _z[:, :, 0], _z[:, :, 1]
read_one_i,  read_one_q  = _o[:, :, 0], _o[:, :, 1]

data_len = read_zero_i.shape[0]
partition_zero, partition_one = 1, 0.25            # normal prior: pulse-driven discrimination
train_div = 0.7
ns_per_sample = 2000 / 4096
st_counters = 8
theta = 0.91
acc_target = 0.90

rz_i = read_zero_i[:int(data_len*partition_zero), :]; rz_q = read_zero_q[:int(data_len*partition_zero), :]
ro_i = read_one_i[:int(data_len*partition_one),  :]; ro_q = read_one_q[:int(data_len*partition_one),  :]
tz1, tz2, ez2 = 0, int(data_len*partition_zero*train_div), int(data_len*partition_zero)
to1, to2, eo2 = 0, int(data_len*partition_one*train_div),  int(data_len*partition_one)


def demod_part(omega, read_i, read_q, phase=0):
    ts = np.arange(0, read_i.shape[1])
    cos_ = np.cos(omega*ts + phase); sin_ = np.sin(omega*ts + phase)
    return np.sum(read_i*cos_ + read_q*sin_, axis=1), np.sum(read_q*cos_ - read_i*sin_, axis=1)


def bayes(prior, obs):
    if obs in (0.0, 1.0): return obs
    return (obs*prior)/(obs*prior + (1-obs)*(1-prior))


def fuse(p, h):
    return (p*h)/(p*h + (1-p)*(1-h)) if p > 0 and h > 0 else max(p, h)


def build_traces(pred_step, window_start):
    def one_set(ai, aq, bi, bq):
        loc_rows, head_rows = [], []
        prev = None
        for k, wl in enumerate(range(pred_step, 2001, pred_step)):
            if window_start + wl > 4096: break
            za = demod_part(omega, ai[:, window_start:window_start+wl], aq[:, window_start:window_start+wl])
            ob = demod_part(omega, bi[:, window_start:window_start+wl], bq[:, window_start:window_start+wl])
            cz = np.array(za).T.mean(axis=0); co = np.array(ob).T.mean(axis=0)
            data = np.c_[np.array(za), np.array(ob)].T
            d = np.linalg.norm(data - cz, axis=1) - np.linalg.norm(data - co, axis=1)
            loc_rows.append((d >= 0).astype(float))
            if k > 0:
                head_rows.append((d >= prev).astype(float))
            prev = d
            n0, n1 = np.array(za).shape[1], np.array(ob).shape[1]
        label = np.array([0]*n0 + [1]*n1)
        return np.vstack(loc_rows), np.vstack(head_rows), label

    mtl = mth = mvl = mvh = None
    for oi, omega in enumerate(omegas):
        tr_loc, tr_head, tr_lab = one_set(rz_i[tz1:tz2], rz_q[tz1:tz2], ro_i[to1:to2], ro_q[to1:to2])
        te_loc, te_head, te_lab = one_set(rz_i[tz2:ez2], rz_q[tz2:ez2], ro_i[to2:eo2], ro_q[to2:eo2])
        if oi == 0:
            mtl, mth, mvl, mvh = tr_loc[..., None], tr_head[..., None], te_loc[..., None], te_head[..., None]
        else:
            mtl = np.concatenate([mtl, tr_loc[..., None]], axis=2)
            mth = np.concatenate([mth, tr_head[..., None]], axis=2)
            mvl = np.concatenate([mvl, te_loc[..., None]], axis=2)
            mvh = np.concatenate([mvh, te_head[..., None]], axis=2)
    return (mtl.transpose(2, 1, 0), mth.transpose(2, 1, 0),
            mvl.transpose(2, 1, 0), mvh.transpose(2, 1, 0), tr_lab, te_lab)


def run_prediction(pred_step, window_start=850):
    trl, trh, tel, teh, train_label, test_label = build_traces(pred_step, window_start)
    BHT_count, HHT_count = {}, {}
    n_steps = trl.shape[2] - st_counters
    for i in range(trl.shape[1]):
        lab = int(train_label[i])
        for j in range(n_steps):
            lk = ''.join('1' if x != 0.0 else '0' for x in trl[0, i, j:j+st_counters])
            c = BHT_count.setdefault(lk, [0, 0]); c[lab] += 1
            if j > 0:
                hk = ''.join('1' if x != 0.0 else '0' for x in trh[0, i, j-1:j+st_counters-1])
                c = HHT_count.setdefault(hk, [0, 0]); c[lab] += 1
    BHT_prob = {k: v[1]/(v[0]+v[1]) for k, v in BHT_count.items()}
    HHT_prob = {k: v[1]/(v[0]+v[1]) for k, v in HHT_count.items()}

    origin = partition_one/(partition_zero+partition_one)
    test_result = np.zeros(tel.shape[1]); latency_ns = np.zeros(tel.shape[1])
    ns = tel.shape[2] - st_counters
    for i in range(tel.shape[1]):
        pred_prob = pred_heading = origin; decided = False
        for j in range(ns):
            lk = ''.join('1' if x != 0.0 else '0' for x in tel[0, i, j:j+st_counters])
            if j > 0:
                hk = ''.join('1' if x != 0.0 else '0' for x in teh[0, i, j-1:j+st_counters-1])
                if hk in HHT_prob: pred_heading = bayes(pred_heading, HHT_prob[hk])
            if lk in BHT_prob: pred_prob = bayes(pred_prob, BHT_prob[lk])
            fused = fuse(pred_prob, pred_heading)
            if not decided and (fused >= theta or fused <= 1 - theta):
                test_result[i] = 0 if fused < 0.5 else 1
                latency_ns[i] = (window_start + (j + st_counters) * pred_step) * ns_per_sample
                decided = True
        if not decided:
            fused = fuse(pred_prob, pred_heading)
            test_result[i] = 0 if fused < 0.5 else 1
            latency_ns[i] = (window_start + (ns - 1 + st_counters) * pred_step) * ns_per_sample
    acc = np.sum(test_result == test_label) / len(test_label)
    return acc, latency_ns.mean()


window_start = 850
ns_list = [20, 30, 40, 50, 60, 80, 100]
rows = []
print(f"{'pred_step(ns)':>13} {'samples':>8} {'accuracy':>9} {'latency(us)':>12}")
for nsec in ns_list:
    ps = int(round(nsec / ns_per_sample))
    acc, lat = run_prediction(ps, window_start)
    rows.append((nsec, ps, acc, lat/1000))
    print(f"{nsec:>13} {ps:>8} {acc:>9.4f} {lat/1000:>12.4f}")

ok = [r for r in rows if r[2] >= acc_target]
best = min(ok, key=lambda r: r[3]) if ok else max(rows, key=lambda r: r[2])
print(f"\n==> best window: pred_step = {best[0]} ns ({best[1]} samples) | acc = {best[2]:.4f} | latency = {best[3]:.4f} us")
print(f"\nAVERAGE FEEDBACK LATENCY = {best[3]:.3f} us")



import matplotlib as mpl
from matplotlib import pyplot as plt
mpl.rcParams['text.usetex'] = False
xs = [r[0] for r in rows]; accs = [r[2] for r in rows]; lats = [r[3] for r in rows]
fig, ax1 = plt.subplots()
ax1.plot(xs, accs, 'o-', color='tab:blue', label='accuracy')
ax1.set_xlabel('window length (ns)'); ax1.set_ylabel('accuracy', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(xs, lats, 's--', color='tab:red', label='feedback latency')
ax2.set_ylabel('feedback latency (us)', color='tab:red')
ax1.axvline(best[0], color='gray', ls=':')
plt.title('window length vs accuracy and feedback latency')
plt.show()


This module **sweeps the demodulation window length** (`pred_step`) and, for each value, runs the full prediction pipeline to measure both **accuracy** and **average feedback latency**. A short window carries too little information (low accuracy); a long window updates less often (higher latency) — so there is an optimal window. The **best window** is chosen as the one with the **lowest latency among those reaching accuracy ≥ 0.90**. It outputs the per-window table, the selected best window, and an accuracy/latency curve.

## 12. Two-Qubit Quantum Random Walk Feedback Test
Apply the same branch-prediction flow to a QRW-style feedback loop where each coin measurement controls the next operation.

The quantum random walk example turns the predictor into a feedback-control primitive. The measured coin qubit determines which conditional operation is applied to the second qubit, so each step contains a measurement-dependent branch.

![Two-qubit quantum random walk feedback circuit](results/5_1_quantum_random_walk_feedback.png)

In hardware terms, the ARTERY output is not only a class label. It selects the feedback branch that drives the next pulse or operation, which is why both prediction accuracy and feedback latency matter.


In [ ]:
import numpy as np
from scipy.io import loadmat

# ===== Two-qubit Quantum Random Walk (QRW) feedback test =====
# QRW circuit: a "coin" qubit and a "position" qubit. Each step the coin is
# measured; the measurement result decides the conditional shift applied to the
# position qubit (the branch / feedback). Over N steps there are N feedbacks,
# so the total feedback latency scales linearly with the number of steps
# (it scales linearly with steps). We predict the coin readout each step with the same
# branch-prediction pipeline, under a QRW-like near-balanced prior (~0.42/0.58).

omegas = 2*np.pi*(np.array([6.881, 6.79525, 6.97284]) - 7)
_rd = loadmat('./readout_data.mat')
_z, _o = _rd['data'][0][:][:][:], _rd['data'][1][:][:][:]
read_zero_i, read_zero_q = _z[:, :, 0], _z[:, :, 1]
read_one_i,  read_one_q  = _o[:, :, 0], _o[:, :, 1]
data_len = read_zero_i.shape[0]
ns_per_sample = 2000 / 4096
train_div = 0.7

partition_zero, partition_one = 1, 0.7      # QRW-like near-balanced prior (P(1) ~= 0.42)
window_start, st_counters, theta = 850, 8, 0.91
pred_step = int(round(100 / ns_per_sample)) # 100 ns demod window

rz_i = read_zero_i[:int(data_len*partition_zero), :]; rz_q = read_zero_q[:int(data_len*partition_zero), :]
ro_i = read_one_i[:int(data_len*partition_one),  :]; ro_q = read_one_q[:int(data_len*partition_one),  :]
tz1, tz2, ez2 = 0, int(data_len*partition_zero*train_div), int(data_len*partition_zero)
to1, to2, eo2 = 0, int(data_len*partition_one*train_div),  int(data_len*partition_one)


def demod_part(omega, ri, rq, phase=0):
    ts = np.arange(0, ri.shape[1]); c = np.cos(omega*ts+phase); s = np.sin(omega*ts+phase)
    return np.sum(ri*c+rq*s, axis=1), np.sum(rq*c-ri*s, axis=1)

def bayes(pr, ob):
    if ob in (0.0, 1.0): return ob
    return (ob*pr)/(ob*pr+(1-ob)*(1-pr))

def fuse(p, h):
    return (p*h)/(p*h+(1-p)*(1-h)) if p > 0 and h > 0 else max(p, h)


def build_traces(pred_step, window_start):
    def one_set(ai, aq, bi, bq):
        lr, hr, prev = [], [], None; n0 = n1 = 0
        for k, wl in enumerate(range(pred_step, 2001, pred_step)):
            if window_start+wl > 4096: break
            za = demod_part(omega, ai[:, window_start:window_start+wl], aq[:, window_start:window_start+wl])
            ob = demod_part(omega, bi[:, window_start:window_start+wl], bq[:, window_start:window_start+wl])
            cz = np.array(za).T.mean(0); co = np.array(ob).T.mean(0)
            data = np.c_[np.array(za), np.array(ob)].T
            d = np.linalg.norm(data-cz, axis=1) - np.linalg.norm(data-co, axis=1)
            lr.append((d >= 0).astype(float))
            if k > 0: hr.append((d >= prev).astype(float))
            prev = d; n0, n1 = np.array(za).shape[1], np.array(ob).shape[1]
        return np.vstack(lr), np.vstack(hr), np.array([0]*n0+[1]*n1)
    mtl=mth=mvl=mvh=None
    for oi, omega in enumerate(omegas):
        a,b,trl = one_set(rz_i[tz1:tz2],rz_q[tz1:tz2],ro_i[to1:to2],ro_q[to1:to2])
        c,d,tel = one_set(rz_i[tz2:ez2],rz_q[tz2:ez2],ro_i[to2:eo2],ro_q[to2:eo2])
        if oi == 0: mtl,mth,mvl,mvh=a[...,None],b[...,None],c[...,None],d[...,None]
        else:
            mtl=np.concatenate([mtl,a[...,None]],2); mth=np.concatenate([mth,b[...,None]],2)
            mvl=np.concatenate([mvl,c[...,None]],2); mvh=np.concatenate([mvh,d[...,None]],2)
    return mtl.transpose(2,1,0),mth.transpose(2,1,0),mvl.transpose(2,1,0),mvh.transpose(2,1,0),trl,tel


def run_prediction(pred_step, window_start, st, theta):
    trl,trh,tel,teh,train_label,test_label = build_traces(pred_step, window_start)
    BHT,HHT={},{}
    for i in range(trl.shape[1]):
        lab=int(train_label[i])
        for j in range(trl.shape[2]-st):
            lk=''.join('1' if x else '0' for x in trl[0,i,j:j+st]); BHT.setdefault(lk,[0,0])[lab]+=1
            if j>0:
                hk=''.join('1' if x else '0' for x in trh[0,i,j-1:j+st-1]); HHT.setdefault(hk,[0,0])[lab]+=1
    BHTp={k:v[1]/(v[0]+v[1]) for k,v in BHT.items()}; HHTp={k:v[1]/(v[0]+v[1]) for k,v in HHT.items()}
    origin=partition_one/(partition_zero+partition_one)
    res=np.zeros(tel.shape[1]); lat=np.zeros(tel.shape[1]); ns=tel.shape[2]-st
    for i in range(tel.shape[1]):
        pp=ph=origin; dec=False
        for j in range(ns):
            lk=''.join('1' if x else '0' for x in tel[0,i,j:j+st])
            if j>0:
                hk=''.join('1' if x else '0' for x in teh[0,i,j-1:j+st-1])
                if hk in HHTp: ph=bayes(ph,HHTp[hk])
            if lk in BHTp: pp=bayes(pp,BHTp[lk])
            f=fuse(pp,ph)
            if not dec and (f>=theta or f<=1-theta):
                res[i]=0 if f<0.5 else 1; lat[i]=(window_start+(j+st)*pred_step)*ns_per_sample; dec=True
        if not dec:
            f=fuse(pp,ph); res[i]=0 if f<0.5 else 1; lat[i]=(window_start+(ns-1+st)*pred_step)*ns_per_sample
    return np.mean(res==test_label), lat.mean()/1000, res, test_label


# ---- per-feedback (single coin measurement) accuracy & latency ----
acc, per_step_lat, qrw_pred, qrw_true = run_prediction(pred_step, window_start, st_counters, theta)
print(f"prior P(1) = {partition_one/(partition_zero+partition_one):.3f}  ")
print(f"per-feedback (one step): accuracy = {acc:.4f}, latency = {per_step_lat:.4f} us\n")

# ---- 2-qubit QRW over N steps: total feedback latency = per-step x #step ----
print(f"{'#step':>5} {'ARTERY(us)':>11} ")
for step in [1, 5, 15, 25]:
    total = per_step_lat * step
    print(f"{step:>5} {total:>11.2f}")

print(f"\nAVERAGE FEEDBACK LATENCY (QRW) = {per_step_lat:.3f} us   ")

from matplotlib import pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = False
steps = np.arange(1, 26)
plt.figure()
plt.plot(steps, per_step_lat*steps, 'o-', color='tab:blue', label='ARTERY (this work)')
plt.xlabel('# step'); plt.ylabel('total feedback latency (us)')
plt.title('two-qubit QRW: feedback latency vs steps')
plt.legend(); plt.show()


**Two-qubit Quantum Random Walk (QRW) test.** A *coin* qubit is measured at every step, and its outcome (the feedback / branch) decides the conditional shift applied to the *position* qubit. Each step is one feedback, so the **total feedback latency grows linearly with the number of steps**. The coin readout is predicted with the same branch-prediction pipeline under a QRW-like **near-balanced prior** (P(1) ≈ 0.42). It outputs the per-feedback accuracy and the average feedback latency, together with the total latency versus number of steps.

## 13. Feedback Waveform Return and Recovery
Generate the branch-dependent feedback pulse, and demonstrate how recovery is applied when the early branch prediction is wrong.

In [ ]:
# ===== Feedback waveform generation (pulse preparation) with recovery =====
# After branch prediction (qrw_pred), the branch decider fetches the gate pulse
# for the predicted branch from the pulse library and sends it back to the qubit
# (DAC). branch 0 -> I (no pulse), branch 1 -> X (pi pulse). If the prediction is
# wrong, recovery is applied: reverse the pre-executed gate (X is self-inverse),
# then apply the correct branch.
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
mpl.rcParams['text.usetex'] = False

dt = 0.5            # ns per DAC sample (2 GSPS) -> 30 ns = 60 samples
gate_len_ns = 30
f_carrier = 0.2     # GHz (200 MHz IF carrier)
branch_gate = {0: 'I', 1: 'X'}

def gen_pulse(gate):
    n = int(round(gate_len_ns / dt)); t = np.arange(n) * dt
    if gate == 'I':
        return np.zeros(n), np.zeros(n)
    sigma = gate_len_ns / 4.0
    env = np.exp(-0.5 * ((t - gate_len_ns / 2) / sigma) ** 2)
    return env * np.cos(2 * np.pi * f_carrier * t), env * np.sin(2 * np.pi * f_carrier * t)

def feedback_waveform(pred, true):
    seq = [branch_gate[int(pred)]]                    # pre-execute predicted branch
    recovered = False
    if int(pred) != int(true):                        # wrong -> recovery
        seq += [branch_gate[int(pred)], branch_gate[int(true)]]   # reverse + correct
        recovered = True
    Is, Qs = [], []
    for g in seq:
        i, q = gen_pulse(g); Is.append(i); Qs.append(q)
    return np.concatenate(Is), np.concatenate(Qs), seq, recovered

# ---- generate feedback waveforms for all QRW test shots ----
n_b0 = n_b1 = n_rec = extra = 0
for p, t in zip(qrw_pred, qrw_true):
    _, _, seq, rec = feedback_waveform(p, t)
    if int(p) == 0: n_b0 += 1
    else: n_b1 += 1
    if rec: n_rec += 1; extra += 2
print(f"shots: {len(qrw_pred)} | branch0(I): {n_b0} | branch1(X): {n_b1}")
print(f"recovery events (wrong predictions): {n_rec} ({n_rec / len(qrw_pred) * 100:.1f}%)")
print(f"extra recovery pulses emitted: {extra}")

# ---- plot 2 examples: correct vs wrong(+recovery) ----
idx_ok = next((k for k in range(len(qrw_pred)) if int(qrw_pred[k]) == int(qrw_true[k]) == 1), 0)
idx_bad = next((k for k in range(len(qrw_pred)) if int(qrw_pred[k]) != int(qrw_true[k])), None)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, idx, title in [(axs[0], idx_ok, 'correct prediction'),
                       (axs[1], idx_bad, 'wrong prediction + recovery')]:
    if idx is None:
        ax.set_title('no such shot'); continue
    I, Q, seq, rec = feedback_waveform(qrw_pred[idx], qrw_true[idx]); t = np.arange(len(I)) * dt
    ax.plot(t, I, color='tab:blue', label='I'); ax.plot(t, Q, color='tab:red', label='Q')
    b = 0
    for g in seq:
        ax.axvline(b * dt, color='gray', ls=':'); ax.text(b * dt + 2, 0.9, g, fontsize=11)
        b += int(round(gate_len_ns / dt))
    ax.set_xlabel('t (ns)'); ax.set_ylabel('amplitude')
    ax.set_title(f"{title}: pred={int(qrw_pred[idx])}, true={int(qrw_true[idx])}, seq={seq}")
    ax.legend()
plt.tight_layout(); plt.show()


After the branch is predicted, the branch decider fetches the corresponding gate pulse from the pulse library and **sends it back to the qubit** (DAC). Mapping: branch 0 → I (no pulse), branch 1 → X (π pulse), synthesised as a Gaussian envelope on a 200 MHz IF carrier (30 ns = 60 samples). If the prediction was wrong, a **recovery** is applied: undo the pre-executed gate (X is self-inverse), then apply the correct branch. It outputs the per-branch counts, the number of recovery events, and example I/Q feedback waveforms (correct vs. wrong + recovery).